In [ ]:
# ============================================================================
# TASK 5: CELL 1 - STRICT ENVIRONMENT & DATA LOADER
# (unchanged from original)
# ============================================================================
import pandas as pd
import numpy as np
import boto3
import io
import warnings
import os
warnings.filterwarnings('ignore')

print('=' * 80)
print('LOADING STRICT TASK 4 TRAINING DATA FROM S3')
print('=' * 80)


def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


USE_CASE_ID = get_use_case_id('pl-aip-uplift')
S3_CONFIG_BUCKET = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
S3_DATA_BUCKET = os.getenv('S3_DATA_BUCKET', 'pratham-nvtabular-poc-data')
EMBEDDING_S3_KEY = 'logical_synthetic_data/processed/vowpal_wabbit_data/MASTER_user_context_vectors.csv'
s3_client = boto3.client('s3')

try:
    print('[1/3] Loading Training Data & Action Library...')
    obj_final = s3_client.get_object(
        Bucket=S3_CONFIG_BUCKET,
        Key=f'training_data/unified_training_{USE_CASE_ID}.parquet'
    )
    df_final = pd.read_parquet(io.BytesIO(obj_final['Body'].read()))

    obj_lib = s3_client.get_object(
        Bucket=S3_CONFIG_BUCKET,
        Key=f'action_library/action_library_{USE_CASE_ID}.parquet'
    )
    action_lib = pd.read_parquet(io.BytesIO(obj_lib['Body'].read()))

    print('[2/3] Loading Demographics & User Features...')
    obj_full = s3_client.get_object(
        Bucket=S3_DATA_BUCKET,
        Key='logical_synthetic_data/processed/raw_splits/full_df.parquet'
    )
    full_df = pd.read_parquet(io.BytesIO(obj_full['Body'].read()))
    if 'master_user_id' in df_final.columns:
        df_final['master_user_id'] = df_final['master_user_id'].astype(str)
    if 'master_user_id' in full_df.columns:
        full_df['master_user_id'] = full_df['master_user_id'].astype(str)

    demo_df = full_df[
        ['master_user_id', 'income_tier', 'city_tier', 'lifecycle_stage', 'risk_level', 'current_cibil']
    ].drop_duplicates('master_user_id')

    product_count_df = (
        full_df.loc[full_df['product_id'].notna() & (full_df['product_id'] != 'unknown'), ['master_user_id', 'product_id']]
        .drop_duplicates()
        .groupby('master_user_id')
        .size()
        .rename('product_count')
        .reset_index()
    )
    demo_df = demo_df.merge(product_count_df, on='master_user_id', how='left')
    demo_df['cibil'] = demo_df['current_cibil'].fillna(720).astype(int)
    demo_df['product_count'] = demo_df['product_count'].fillna(0).astype(int)
    demo_df = demo_df.drop(columns=['current_cibil'])
    demo_lookup = demo_df.set_index('master_user_id').to_dict('index')

    embedding_cols = [f'v{i}' for i in range(64)]
    available_embedding_cols = [c for c in embedding_cols if c in full_df.columns]

    if len(available_embedding_cols) == len(embedding_cols):
        df_merged = (
            full_df[['master_user_id'] + embedding_cols]
            .drop_duplicates('master_user_id')
            .set_index('master_user_id')
        )
    else:
        print(f'  Loading learned context vectors from s3://{S3_DATA_BUCKET}/{EMBEDDING_S3_KEY}')
        obj_embed = s3_client.get_object(Bucket=S3_DATA_BUCKET, Key=EMBEDDING_S3_KEY)
        embed_df = pd.read_csv(io.BytesIO(obj_embed['Body'].read()))
        id_candidates = ['master_user_id', 'customer_id', 'user_id']
        id_col = next((c for c in id_candidates if c in embed_df.columns), None)
        if id_col is None:
            raise ValueError(
                f'Embedding file is missing a user id column. Expected one of {id_candidates}, found {list(embed_df.columns[:10])}.'
            )
        csv_embedding_cols = [c for c in embedding_cols if c in embed_df.columns]
        if len(csv_embedding_cols) != len(embedding_cols):
            raise ValueError(
                f'Embedding file must contain 64 vector columns named v0..v63. Found {len(csv_embedding_cols)} matching columns.'
            )
        df_merged = (
            embed_df[[id_col] + embedding_cols]
            .drop_duplicates(id_col)
            .rename(columns={id_col: 'master_user_id'})
            .set_index('master_user_id')
        )

    print(f'\nRECOVERY COMPLETE')
    print(f'  df_final rows     : {len(df_final):,}')
    print(f'  action_lib rows  : {len(action_lib):,}')
    print(f'  users (demo)     : {len(demo_lookup):,}')
    print(f'  embeddings shape : {df_merged.shape}')

except Exception as e:
    print(f'\nRECOVERY FAILED: {str(e)}')
    raise

print('=' * 80)


In [ ]:
# ============================================================================
# TASK 5: CELL 2 - COLD-START OR STRICT LOGGED SLATES
# (unchanged from original — propensity validation logic is correct)
# ============================================================================
from collections import defaultdict
import json
import os
import pandas as pd

print('=' * 80)
print('TASK 5: VALIDATING TRAINING SLATES AND PROPENSITIES')
print('=' * 80)

REQUESTED_TRAINING_MODE = os.getenv('TRAINING_MODE', '').strip().lower()
NUM_ACTIONS_PER_SLATE = 60
MIN_ACTIONS_PER_SLATE = 50
PROPENSITY_COLS = ['chosen_action_prob', 'logged_prob', 'logging_prob', 'propensity']
ACTION_FEATURE_COLS = ['channel', 'day', 'time', 'offer']

missing_action_feature_cols = [c for c in ACTION_FEATURE_COLS if c not in action_lib.columns]
if missing_action_feature_cols:
    raise ValueError(f'Action library missing required columns for VW action features: {missing_action_feature_cols}')
action_lib['action_id'] = action_lib['action_id'].astype(int)
action_features_dict = action_lib.set_index('action_id')[ACTION_FEATURE_COLS].to_dict('index')
valid_action_ids = set(int(aid) for aid in action_features_dict.keys())

df_valid = df_final[df_final['action_id'].astype(int).isin(valid_action_ids)].copy()
if df_valid.empty:
    raise ValueError('No valid training rows remain after action-library filtering.')

required_training_cols = ['decision_id', 'candidate_slate_json', 'action_id', 'vw_cost']
missing_required_training_cols = [c for c in required_training_cols if c not in df_valid.columns]
if missing_required_training_cols:
    raise ValueError(f'Task 5 input missing required columns: {missing_required_training_cols}. Run Task 4 first.')

available_propensity_cols = [c for c in PROPENSITY_COLS if c in df_valid.columns]
if not available_propensity_cols:
    raise ValueError(f'Task 5 requires one propensity column. Expected one of {PROPENSITY_COLS}.')

resolved_probs = []
resolved_sources = []
missing_propensity_rows = []
for idx, row in df_valid.iterrows():
    logged_prob = None
    prob_source = None
    for col in available_propensity_cols:
        value = row.get(col)
        if pd.isna(value):
            continue
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue
        if 0.0 < value <= 1.0:
            logged_prob = value
            source_override = row.get('propensity_source') if 'propensity_source' in df_valid.columns else None
            if pd.notna(source_override) and str(source_override).strip():
                prob_source = str(source_override).strip()
            else:
                prob_source = f'column:{col}'
            break
    if logged_prob is None:
        missing_propensity_rows.append(idx)
        resolved_probs.append(None)
        resolved_sources.append(None)
    else:
        resolved_probs.append(float(logged_prob))
        resolved_sources.append(prob_source)

if missing_propensity_rows:
    sample = df_valid.loc[
        missing_propensity_rows[:10],
        [c for c in ['decision_id', 'campaign_id', 'master_user_id', 'timestamp', 'action_id'] if c in df_valid.columns]
    ]
    raise ValueError(f'Found {len(missing_propensity_rows):,} rows without valid propensity.\n' + sample.to_string(index=False))

df_valid['logged_prob'] = resolved_probs
df_valid['prob_source'] = resolved_sources
policies = set(df_valid.get('decision_policy', pd.Series(dtype=str)).dropna().astype(str).str.lower())
source_counts = pd.Series(resolved_sources).value_counts()
source_counts_dict = {str(k): int(v) for k, v in source_counts.items()}
contains_synthetic_propensity = any(not str(k).startswith('column:') for k in source_counts_dict)
if REQUESTED_TRAINING_MODE:
    TRAINING_MODE = REQUESTED_TRAINING_MODE
else:
    TRAINING_MODE = 'cold_start_uniform' if contains_synthetic_propensity or 'cold_start_uniform' in policies else 'strict_ope'
if TRAINING_MODE not in {'strict_ope', 'cold_start_uniform'}:
    raise ValueError("TRAINING_MODE must be 'strict_ope' or 'cold_start_uniform'.")

strict_ope_ready = bool(source_counts_dict) and (not contains_synthetic_propensity) and TRAINING_MODE == 'strict_ope'
if TRAINING_MODE == 'strict_ope' and not strict_ope_ready:
    raise ValueError(f'strict_ope requested but non-logged propensity sources were found: {source_counts_dict}')


def parse_logged_slate(value, row_id):
    if isinstance(value, list):
        slate = value
    else:
        if pd.isna(value):
            raise ValueError(f'candidate_slate_json is null for row {row_id}.')
        try:
            slate = json.loads(value)
        except Exception as exc:
            raise ValueError(f'candidate_slate_json is invalid JSON for row {row_id}.') from exc
    if not isinstance(slate, list):
        raise ValueError(f'candidate_slate_json must decode to a list for row {row_id}.')
    try:
        slate = [int(aid) for aid in slate]
    except Exception as exc:
        raise ValueError(f'candidate_slate_json has non-integer action ids for row {row_id}.') from exc
    if len(slate) < MIN_ACTIONS_PER_SLATE:
        raise ValueError(f'Slate too small for row {row_id}: {len(slate)} < {MIN_ACTIONS_PER_SLATE}.')
    if len(set(slate)) != len(slate):
        raise ValueError(f'Slate contains duplicate action ids for row {row_id}.')
    unknown = [aid for aid in slate if aid not in valid_action_ids]
    if unknown:
        raise ValueError(f'Slate contains unknown action ids for row {row_id}: {unknown[:10]}')
    return slate


seen_decision_ids = set()
event_action_spaces = []
event_key_counts = defaultdict(int)
training_records = df_valid.to_dict('records')
print(f'Validating slates for {len(training_records):,} events...')
for idx, row in enumerate(training_records):
    decision_id = str(row['decision_id'])
    if decision_id in seen_decision_ids:
        raise ValueError(f'Duplicate decision_id found in Task 5 input: {decision_id}')
    seen_decision_ids.add(decision_id)
    uid = str(row['master_user_id'])
    real_action = int(row['action_id'])
    slate = parse_logged_slate(row['candidate_slate_json'], decision_id)
    if real_action not in slate:
        raise ValueError(f'Chosen action {real_action} missing from slate for decision_id={decision_id} user={uid}.')
    event_key = decision_id
    event_key_counts[event_key] += 1
    event_action_spaces.append({
        'decision_id': decision_id,
        'campaign_id': row.get('campaign_id'),
        'master_user_id': uid,
        'event_order': int(event_key_counts[event_key]),
        'taken_action': real_action,
        'vw_cost': float(row['vw_cost']),
        'logged_prob': float(row['logged_prob']),
        'prob_source': row['prob_source'],
        'slate': slate,
        'candidate_slate_json': json.dumps(slate),
    })
    if (idx + 1) % 5000 == 0 or (idx + 1) == len(training_records):
        print(f'  -> Progress: {idx + 1:,} / {len(training_records):,} slates validated.')

propensity_audit = {
    'use_case_id': USE_CASE_ID,
    'training_mode': TRAINING_MODE,
    'total_events': int(len(event_action_spaces)),
    'unique_decision_ids': int(len(seen_decision_ids)),
    'duplicate_event_keys': int(sum(1 for cnt in event_key_counts.values() if cnt > 1)),
    'source_counts': source_counts_dict,
    'has_true_logged_propensity': bool(strict_ope_ready),
    'has_non_logged_propensity': bool(contains_synthetic_propensity),
    'has_logged_candidate_slates': bool(strict_ope_ready),
    'has_synthetic_candidate_slates': bool(TRAINING_MODE == 'cold_start_uniform'),
    'min_slate_size': int(min(len(e['slate']) for e in event_action_spaces)),
    'max_slate_size': int(max(len(e['slate']) for e in event_action_spaces)),
    'strict_ope_ready': bool(strict_ope_ready),
    'production_promotable': False,
}

propensity_audit_key = f'training_data/vw_propensity_audit_{USE_CASE_ID}.json'
s3_client.put_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=propensity_audit_key,
    Body=json.dumps(propensity_audit, indent=2).encode('utf-8')
)

print('=' * 80)
print('TRAINING SLATES VALIDATED')
print(f'Training mode      : {TRAINING_MODE}')
print(f'Total events       : {len(event_action_spaces):,}')
print(f'Unique decisions   : {len(seen_decision_ids):,}')
print(f'Propensity sources : {source_counts_dict}')
print(f'Strict OPE ready   : {strict_ope_ready}')
print(f'Slate size range   : {propensity_audit["min_slate_size"]} - {propensity_audit["max_slate_size"]}')
print(f'Audit uploaded     : s3://{S3_CONFIG_BUCKET}/{propensity_audit_key}')
print('=' * 80)


In [ ]:
# ====================================================================================
# TASK 5: CELL 3 - GENERATING PRODUCTION STRICT VW DATA
#
# BUG 6 FIX: VW cb_adf label format for the CHOSEN action must be:
#   0:{cost}:{prob:.6f} |action ...
# NOT:
#   {cost}:{prob:.6f} |action ...
#
# The '0:' prefix is the action index (always 0 for cb_adf since the chosen
# action is always placed first in the block and VW ignores this field).
# Without it, VW treats the line as a namespace line not a labeled example,
# causing: 'cb_adf: badly formatted example, only one line can have a cost'.
#
# This is the canonical format per VW wiki:
#   https://github.com/VowpalWabbit/vowpal_wabbit/wiki/Contextual-Bandit-algorithms
#   "action_idx:cost:prob |action feats"
#
# BUG 7 FIX: LABEL_RE must match this same format to validate the output.
# ====================================================================================
import io
import re

# BUG 7 FIX: LABEL_RE updated to require '0:' prefix (canonical cb_adf format)
# This matches what we write below AND what Task 6/7 must parse.
LABEL_RE = re.compile(
    r'^0:(?P<cost>[0-9]*\.?[0-9]+):(?P<prob>[0-9]*\.?[0-9]+)\s+\|action\b'
)

vw_blocks = []
aligned_rows = []
missing_event_rows = []
missing_embedding_rows = []

event_lookup = {}
for e in event_action_spaces:
    if e['decision_id'] in event_lookup:
        raise ValueError(f'Duplicate event for decision_id={e["decision_id"]}')
    event_lookup[e['decision_id']] = e

source_df = df_valid.reset_index(drop=False).rename(columns={'index': 'source_row_index'})

for _, row in source_df.iterrows():
    decision_id = str(row['decision_id'])
    campaign_id = row.get('campaign_id', 'NA')
    uid = row['master_user_id']

    if uid not in df_merged.index:
        missing_embedding_rows.append((decision_id, uid))
        continue

    event = event_lookup.pop(decision_id, None)
    if event is None:
        missing_event_rows.append((decision_id, uid))
        continue

    emb = df_merged.loc[uid].values
    emb_str = ' '.join([f'v{i}:{round(v, 6)}' for i, v in enumerate(emb)])
    demo = demo_lookup.get(uid, {})
    user_str = (
        f"inc={demo.get('income_tier', 'NA')} "
        f"city={demo.get('city_tier', 'NA')} "
        f"life={demo.get('lifecycle_stage', 'NA')} "
        f"risk={demo.get('risk_level', 'NA')} "
        f"cibil:{demo.get('cibil', 0)} "
        f"products:{demo.get('product_count', 0)}"
    )

    taken_action = int(event['taken_action'])
    cost = round(float(event['vw_cost']), 4)
    prob = min(max(float(event['logged_prob']), 1e-6), 1.0)
    slate = [int(aid) for aid in event['slate']]
    if taken_action not in slate:
        raise ValueError(f'Taken action {taken_action} missing from slate for decision_id={decision_id} user={uid}.')
    if not (0.0 <= cost <= 1.0):
        raise ValueError(f'VW cost out of bounds for decision_id={decision_id} user={uid}: {cost}')
    if not (0.0 < prob <= 1.0):
        raise ValueError(f'Logged probability out of bounds for decision_id={decision_id} user={uid}: {prob}')

    block_lines = [f'shared |emb {emb_str} |user {user_str}']
    feats = action_features_dict.get(taken_action, {})

    # BUG 6 FIX: Use canonical cb_adf label format: '0:{cost}:{prob} |action ...'
    # The '0:' is the required action-index prefix. Without it VW cannot parse the label.
    block_lines.append(
        f'0:{cost}:{prob:.6f} |action aid_{taken_action} '
        f'ch={feats.get("channel", "NA")} '
        f'dw={feats.get("day", "NA")} '
        f'tm={feats.get("time", "NA")} '
        f'of={feats.get("offer", "NA")} '
    )

    for aid in slate:
        if aid == taken_action:
            continue
        feats = action_features_dict.get(aid, {})
        block_lines.append(
            f'|action aid_{aid} '
            f'ch={feats.get("channel", "NA")} '
            f'dw={feats.get("day", "NA")} '
            f'tm={feats.get("time", "NA")} '
            f'of={feats.get("offer", "NA")} '
        )

    # Validate the block we just built
    label_count = sum(1 for line in block_lines[1:] if LABEL_RE.match(line))
    if label_count != 1 or not LABEL_RE.match(block_lines[1]):
        raise ValueError(f'VW block label validation failed for decision_id={decision_id} user={uid}.')
    if len(block_lines) - 1 != len(slate):
        raise ValueError(f'VW action count/slate mismatch for decision_id={decision_id} user={uid}.')

    vw_blocks.append('\n'.join(block_lines))
    aligned_record = row.to_dict()
    aligned_record.update({
        'decision_id': decision_id,
        'campaign_id': campaign_id,
        'event_order': int(event['event_order']),
        'taken_action': taken_action,
        'logged_prob': float(prob),
        'prob_source': event.get('prob_source'),
        'slate_size': int(len(slate)),
        'candidate_slate_json': event.get('candidate_slate_json'),
    })
    aligned_rows.append(aligned_record)

if missing_embedding_rows:
    raise ValueError(f'Missing embeddings for {len(missing_embedding_rows):,} rows. First examples: {missing_embedding_rows[:5]}')
if missing_event_rows:
    raise ValueError(f'Missing event records for {len(missing_event_rows):,} rows. First examples: {missing_event_rows[:5]}')
remaining_events = len(event_lookup)
if remaining_events:
    raise ValueError(f'Unconsumed event records after VW generation: {remaining_events:,}')
if len(vw_blocks) != len(df_valid):
    raise ValueError(f'VW block count mismatch: blocks={len(vw_blocks):,}, rows={len(df_valid):,}')

s3_vw_key = f'training_data/vw_training_{USE_CASE_ID}_FINAL.txt'
alignment_key = f'training_data/vw_training_alignment_{USE_CASE_ID}.parquet'
s3_client.put_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=s3_vw_key,
    Body=('\n\n'.join(vw_blocks) + '\n').encode('utf-8')
)

aligned_df = pd.DataFrame(aligned_rows)
alignment_buffer = io.BytesIO()
aligned_df.to_parquet(alignment_buffer, index=False)
s3_client.put_object(Bucket=S3_CONFIG_BUCKET, Key=alignment_key, Body=alignment_buffer.getvalue())

print('=' * 80)
print('PRODUCTION VW TRAINING DATA GENERATED SUCCESSFULLY')
print(f'Uploaded to: s3://{S3_CONFIG_BUCKET}/{s3_vw_key}')
print(f'Total blocks: {len(vw_blocks):,}')
print(f'Alignment uploaded: s3://{S3_CONFIG_BUCKET}/{alignment_key}')
print('BUG 6 FIX: label format is now canonical cb_adf: 0:{cost}:{prob} |action ...')
print('=' * 80)


In [ ]:
# ============================================================================
# INSPECT THE FINAL PRODUCTION VW FILE (FIRST TWO BLOCKS)
# ============================================================================
print('=' * 80)
print('INSPECTING FINAL VW TRAINING FILE (PRODUCTION FORMAT)')
print('=' * 80)

s3_vw_key = f'training_data/vw_training_{USE_CASE_ID}_FINAL.txt'

try:
    obj = s3_client.get_object(Bucket=S3_CONFIG_BUCKET, Key=s3_vw_key)
    content = obj['Body'].read().decode('utf-8')
    lines = content.splitlines()

    blocks_printed = 0
    for line in lines:
        print(line.strip())
        if line.strip() == '':
            blocks_printed += 1
            if blocks_printed >= 2:
                break

    print('=' * 80)
    print(f'Inspection of {blocks_printed} blocks complete from s3://{S3_CONFIG_BUCKET}/{s3_vw_key}')

except Exception as e:
    print(f'ERROR reading S3 file: {str(e)}')
